# Lesson 3: Building an Agent Reasoning Loop

## Setup

In [1]:
import importlib
import helper
import os
importlib.reload(helper)
from helper import get_openai_api_key, get_dashscope_api_key, get_github_token

import nest_asyncio
# 应用异步支持（在Jupyter环境中必需）
nest_asyncio.apply()

## Load the data

## Setup the Query Tools And Setup Function Calling Agent

In [2]:
from llama_index.llms.openai import OpenAI
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# 配置全局设置：指定使用的语言模型和嵌入模型
# 这里用了DashScope的大模型替代OpenAI模型，
# LlamaIndex支持多种LLM接口，DaskScope兼容OpenAI API，可以使用OpenAILike类调用
# LlamaIndex也有专门的DashScope支持包，具体见 https://developers.llamaindex.ai/python/examples/llm/dashscope/
llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)

# 通过Azure OpenAI调用OpenAI模型,
# 用这个方法需要把helper.py中的get_github_token方法返回的key换成之前在github上申请的token
# llm = OpenAILike(
#     api_key=get_github_token(),
#     api_base="https://models.inference.ai.azure.com/",
#     model="gpt-4o-mini",
#     temperature=0.1,
#     context_window=128000,
#     is_chat_model=True,
#     is_function_calling_model=True,
# )


Settings.llm = llm
# 需要openai key所以改成开源模型
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
Settings.embed_model = HuggingFaceEmbedding(
        model_name=model_real_path,
        # 默认情况下，LlamaIndex 会尝试自动下载和加载模型
        device="cpu",  # 如果您没有GPU，可以使用"cpu"
        # 连不了外网下载模型到本地的记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
        local_files_only=True,
    )


In [3]:

from utils import get_doc_tools

vector_tool, summary_tool = get_doc_tools("metagpt.pdf", "metagpt")

In [ ]:
# 从 LlamaIndex 的核心代理工作流模块中导入 FunctionAgent 类
from llama_index.core.agent.workflow import FunctionAgent

# --- 1. 初始化 FunctionAgent 实例 ---
agent = FunctionAgent(
    # tools: 为代理挂载工具集。
    # vector_tool: 通常用于针对文档特定细节的精准检索（RAG）。
    # summary_tool: 通常用于对整篇文档或大量信息的宏观总结。
    tools=[vector_tool, summary_tool], 
    # llm: 指定代理使用的“大脑”，即之前定义好的大语言模型实例。
    llm=llm, 
    # verbose=True: 开启详细模式。
    # 代理在思考过程中的每一步（如决策调用哪个工具、观察到的结果等）都会在控制台打印输出。
    verbose=True
)

# --- 2. 异步执行任务 ---
# 使用 await 异步运行代理，并向其发送关于 MetaGPT 的查询指令。
# agent.run 方法会启动内部工作流：
#   1. 语义分析用户的意图。
#   2. 决定先调用 summary_tool 获取角色概览，还是调用 vector_tool 查询通信细节。
#   3. 整合工具返回的结果并生成最终答案。
response = await agent.run(
    user_msg="Tell me about the agent roles in MetaGPT, and then how they communicate with each other."
)

2026-05-05 17:32:01,366 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the agent roles in MetaGPT, and then...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-05 17:32:01,366 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-05 17:32:01,369 - INFO - [init_run:0] complete with AgentInput
2026-05-05 17:32:01,370 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT, and then how they communica...
2026-05-05 17:32:01,370 - INFO - [setup_agent:0] started from AgentInput
2026-05-05 17:32:01,371 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-05 17:32:01,372 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT, and then how they comm

2026-05-05 17:56:37,604 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-05 17:59:02,971 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-05 18:06:22,792 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-05 18:06:24,261 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


In [6]:
# 遍历 Agent 响应中包含的所有工具调用（Tool Calls）
for tool_call in response.tool_calls:
    
    # 检查该工具调用的原始输出对象（raw_output）是否具有名为 "source_nodes" 的属性
    # 在 LlamaIndex 的 RAG 工具中，检索到的文档块通常存储在 source_nodes 中
    if hasattr(tool_call.tool_output.raw_output, "source_nodes"):
        
        # 如果存在源节点，则遍历 raw_output 中存储的每一个节点（n）
        # 每个节点通常是一个包含文本和元数据的 NodeWithScore 对象
        for n in tool_call.tool_output.raw_output.source_nodes:
            
            # 打印该节点的元数据字典
            # 这通常包含文件名、页码、文件路径或其他你在索引阶段注入的自定义信息
            print(n.metadata)

print(response)

{'page_label': '4', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
{'page_label': '9', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
{'page_label': '6', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
{'page_label': '4', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
In MetaGPT, the simulated software company structure includes five specialized roles:

- **Product Manager**: Responsible for business-oriented analysis and deriving insights.
- **Architect**: Designs the sys

In [7]:
# 从 LlamaIndex 的工作流模块中导入 Context 类
# Context 是 Workflow 的核心，负责在不同步骤（Steps）之间传递和存储状态
from llama_index.core.workflow import Context

# 为指定的 agent 实例化一个上下文对象
# 这相当于给 Agent 分配了一个“共享存储空间”，用于记录运行时的变量和历史信息
ctx = Context(agent)

# 此代码为了演示如何实现上下文传递
# 使用 await 异步调用 agent.run 方法，并传入用户问题
# 关键点：传入 ctx=ctx 后，Agent 在执行过程中产生的所有中间状态（如工具输出、推理链）都会保留在这个上下文中
response = await agent.run(
    user_msg="Tell me about the evaluation datasets used.", 
    ctx=ctx
)

# 遍历 response 对象中的 tool_calls 列表
# 这可以让你看到 Agent 为了回答问题，具体调用了后台的哪些工具
for tool_call in response.tool_calls:
    # 打印工具的名称以及该工具返回的详细输出
    # 注：代码中的“掉”字应为“调”（调用）的笔误，打印出工具调用的轨迹
    print(f"调{tool_call.tool_name}返回:{tool_call.tool_output}")
    
# 打印装饰线，用于区分工具日志和最终回答
print("\n=== Agent Response ===")

# 打印 Agent 最终生成的回复字符串
# 这是 Agent 整合了所有工具调用结果后，给出的经过润色的自然语言答案
print(str(response))

2026-05-05 17:56:27,434 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation datasets used.', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-05 17:56:27,435 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-05 17:56:27,441 - INFO - [init_run:0] complete with AgentInput
2026-05-05 17:56:27,441 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation datasets used.')])], current_agent_name=...
2026-05-05 17:56:27,441 - INFO - [setup_agent:0] started from AgentInput
2026-05-05 17:56:27,442 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-05 17:56:27,443 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation datasets used.')])], current_agent_name=...
2

调vector_tool_metagpt返回:The evaluation datasets used include:

- **HumanEval**: This dataset contains 164 handwritten programming tasks, which come with function specifications, descriptions, reference codes, and tests.

- **MBPP (Mostly Basic Python Problems)**: It consists of 427 Python tasks that cover core concepts and standard library features. These tasks also include descriptions, reference codes, and automated tests.

- **SoftwareDev**: This is a collection of 70 representative examples of software development tasks. Each task has its own prompt and covers a wide range of scopes such as mini-games, image processing algorithms, and data visualization. The focus of this dataset is on the engineering aspects of software development.

=== Agent Response ===
The evaluation datasets used in the context of the MetaGPT paper include:

- **HumanEval**: A dataset that consists of 164 handwritten programming tasks. These tasks are accompanied by function specifications, descriptions, refer

In [8]:
# 此问题要知道上面有哪些数据集，说以必须有上下文才能回答
response = await agent.run(user_msg="Tell me the results over one of the above datasets.", ctx=ctx)  # 关于上述数据集之一的结果。

for tool_call in response.tool_calls:
    print(f"调{tool_call.tool_name}返回:{tool_call.tool_output}")
print("\n=== Agent Response ===")
print(str(response))

2026-05-05 17:58:58,147 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me the results over one of the above datasets...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-05 17:58:58,147 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-05 17:58:58,149 - INFO - [init_run:0] complete with AgentInput
2026-05-05 17:58:58,150 - INFO - [tick] add: AgentInput(input=[5 items], current_agent_name='Agent')
2026-05-05 17:58:58,150 - INFO - [setup_agent:0] started from AgentInput
2026-05-05 17:58:58,151 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-05 17:58:58,152 - INFO - [tick] add: AgentSetup(input=[5 items], current_agent_name='Agent')
2026-05-05 17:58:58,152 - INFO - [run_agent_step:0] started from AgentSetup
2026-05-05 17:58:59,123 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-05 17:58:59,561 - INFO - [run_agent_step:0] complete with Ag

调vector_tool_metagpt返回:MetaGPT achieves a Pass @1 rate of 85.9% on the HumanEval dataset, outperforming other approaches. When collaborating with GPT-4, it further improves the Pass @k in the HumanEval benchmark.

=== Agent Response ===
On the HumanEval dataset, MetaGPT achieves a Pass @1 rate of 85.9%, outperforming other approaches. Furthermore, when collaborating with GPT-4, it further improves the Pass @k in the HumanEval benchmark.


## Lower-Level: Debuggability and Control（较低级别：可调试性和控制）

注意：在新版本 LlamaIndex 中，原有的 agent.create_task 和 agent.run_step(task.task_id) 逐步执行任务的 API 已被重构。现在推荐的做法是直接使用 agent.run 或 agent.astream_chat/stream_chat 进行任务执行和流式事件处理，底层会自动管理 step-wise 执行和事件流。你可以通过异步 for event in handler.stream_events() 方式获取每一步的事件，包括工具调用、LLM输出等，达到逐步追踪和控制的效果。新版本里没法像视频里说的执行一步停下来，再手动执行下一步，

In [9]:
from llama_index.core.agent.workflow import ToolCallResult, AgentOutput, AgentInput, AgentStream, ToolCall 
from llama_index.core.workflow import Context, InputRequiredEvent, HumanResponseEvent

ctx = Context(agent)  # 创建上下文

agent = FunctionAgent(
    tools=[vector_tool, summary_tool], 
    llm=llm, 
    verbose=True
)

handler = agent.run(user_msg="Tell me about the agent roles in MetaGPT,  and then how they communicate with each other.", ctx=ctx)

completed_steps = []
i=1
async for event in handler.stream_events():
    completed_steps.append(event)
    print(f"\n--- Step {i} Event: {type(event).__name__} ---")
    i+=1
    if isinstance(event, AgentStream):
        print(event.delta, end="", flush=True)
    elif isinstance(event, AgentInput):
        print("|Agent input: ", event.input)  # the current input messages
        print("|Agent name:", event.current_agent_name)  # the current agent name
    elif isinstance(event, AgentOutput):
        print("|Agent output: ", event.response)  # the current full response
        print("|Tool calls made: ", event.tool_calls)  # the selected tool calls, if any
        print("|Raw LLM response: ", event.raw)  # the raw llm api response
    elif isinstance(event, ToolCallResult):
        print("|Tool called: ", event.tool_name)  # the tool name
        print("|Arguments to the tool: ", event.tool_kwargs)  # the tool kwargs
        print("|Tool output: ", event.tool_output)  # the tool output     
print(f"Num completed: {len(completed_steps)}")

print("\n=== Final Result ===")
final_result = await handler
print(final_result)

2026-05-05 18:05:59,646 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the agent roles in MetaGPT,  and the...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-05 18:05:59,647 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-05 18:05:59,650 - INFO - [init_run:0] complete with AgentInput
2026-05-05 18:05:59,651 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT,  and then how they communic...
2026-05-05 18:05:59,651 - INFO - [setup_agent:0] started from AgentInput
2026-05-05 18:05:59,653 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-05 18:05:59,653 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT,  and then how they com


--- Step 1 Event: AgentInput ---
|Agent input:  [ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the agent roles in MetaGPT,  and then how they communicate with each other.')])]
|Agent name: Agent


2026-05-05 18:06:03,423 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"



--- Step 2 Event: AgentStream ---

--- Step 3 Event: AgentStream ---

--- Step 4 Event: AgentStream ---

--- Step 5 Event: AgentStream ---

--- Step 6 Event: AgentStream ---

--- Step 7 Event: AgentStream ---

--- Step 8 Event: AgentStream ---

--- Step 9 Event: AgentStream ---

--- Step 10 Event: AgentStream ---

--- Step 11 Event: AgentStream ---

--- Step 12 Event: AgentStream ---


2026-05-05 18:06:09,892 - INFO - [run_agent_step:0] complete with AgentOutput
2026-05-05 18:06:09,893 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaToolCall(index=0, id='call_71d253b95b99438685f73d', function=ChoiceDeltaTool...
2026-05-05 18:06:09,893 - INFO - [parse_agent_output:0] started from AgentOutput
2026-05-05 18:06:09,896 - INFO - [tick] add: ToolCall(tool_name='vector_tool_metagpt', tool_kwargs={'query': 'agent roles in MetaGPT', 'page_numbers': None}, tool_id='call_71d253b95b99438685f73d')
2026-05-05 18:06:09,897 - INFO - [call_tool:0] started from ToolCall
2026-05-05 18:06:09,902 - INFO - [parse_agent_output:0] complete with no result
2026-05-05 18:06:09,908 - INFO - [tick] add: ToolCall(tool_name='vector_tool_metagpt', tool_kwargs={'query': 'how agents communicate with each other in MetaGPT', 'page_numbers': None}, tool_id='call_8186d030d490447f812d48')
2026-05-05 18:06:09,909 -

...


--- Step 17 Event: ToolCallResult ---
|Tool called:  vector_tool_metagpt
|Arguments to the tool:  {'query': 'how agents communicate with each other in MetaGPT', 'page_numbers': None}
|Tool output:  In MetaGPT, agents communicate with each other through a structured communication approach that involves the use of documents and diagrams rather than direct dialogue. They utilize a shared message pool where they can publish and subscribe to messages based on their specific roles and needs. This method ensures that all necessary information is contained within these structured outputs, which helps in preventing irrelevant or missing content. Each agent can directly retrieve the required information from the shared pool, streamlining the communication process and enhancing efficiency. Additionally, there's a subscription mechanism that allows agents to filter and receive only the information relevant to their tasks, thereby avoiding information overload.

--- Step 18 Event: AgentInput ---
|

2026-05-05 18:06:25,185 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"



--- Step 19 Event: AgentStream ---
In MetaG
--- Step 20 Event: AgentStream ---
PT, the
--- Step 21 Event: AgentStream ---
 agent roles are
--- Step 22 Event: AgentStream ---
 structured like a simulated
--- Step 23 Event: AgentStream ---
 software company and include
--- Step 24 Event: AgentStream ---
 the following:

-
--- Step 25 Event: AgentStream ---
 **Product Manager**: Focuses
--- Step 26 Event: AgentStream ---
 on business-oriented analysis and deriving
--- Step 27 Event: AgentStream ---
 insights.
- **Architect
--- Step 28 Event: AgentStream ---
**: Designs the overall
--- Step 29 Event: AgentStream ---
 system architecture.
-
--- Step 30 Event: AgentStream ---
 **Project Manager**: Man
--- Step 31 Event: AgentStream ---
ages the project timeline
--- Step 32 Event: AgentStream ---
, resources, and
--- Step 33 Event: AgentStream ---
 ensures that tasks are completed.
--- Step 34 Event: AgentStream ---

- **Engineer**: Responsible
--- Step 35 Event: AgentStream ---
 for program

2026-05-05 18:06:57,829 - INFO - [run_agent_step:0] complete with AgentOutput
2026-05-05 18:06:57,829 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='In MetaGPT, the agent roles are structured like a simula...
2026-05-05 18:06:57,830 - INFO - [parse_agent_output:0] started from AgentOutput
2026-05-05 18:06:57,834 - INFO - [result] StopEvent(result=AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='In MetaGPT, the agent roles are structu...
2026-05-05 18:06:57,834 - INFO - [parse_agent_output:0] complete with StopEvent



--- Step 67 Event: AgentOutput ---
|Agent output:  assistant: In MetaGPT, the agent roles are structured like a simulated software company and include the following:

- **Product Manager**: Focuses on business-oriented analysis and deriving insights.
- **Architect**: Designs the overall system architecture.
- **Project Manager**: Manages the project timeline, resources, and ensures that tasks are completed.
- **Engineer**: Responsible for programming and can execute code.
- **QA Engineer**: Ensures the quality of the product through testing and validation.

These agents work in a coordinated manner, adhering to standard operating procedures (SOPs) to manage their tasks. They do not communicate via direct dialogue but rather through a shared message pool. This pool allows them to publish and subscribe to messages relevant to their functions, ensuring that all necessary information is contained within these structured outputs.

The communication between agents is further streamlined by 

In [ ]:
# 没有加await,这样就能异步掉启，不用等待结果返回，直接执行下面的代码
handler =  agent.run(user_msg="What about how agents share information?", ctx=ctx)

2026-05-05 18:09:32,773 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='What about how agents share information?', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-05 18:09:32,773 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-05 18:09:32,775 - INFO - [init_run:0] complete with AgentInput
2026-05-05 18:09:32,776 - INFO - [tick] add: AgentInput(input=[6 items], current_agent_name='Agent')
2026-05-05 18:09:32,776 - INFO - [setup_agent:0] started from AgentInput
2026-05-05 18:09:32,777 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-05 18:09:32,778 - INFO - [tick] add: AgentSetup(input=[6 items], current_agent_name='Agent')
2026-05-05 18:09:32,778 - INFO - [run_agent_step:0] started from AgentSetup


2026-05-05 18:09:33,906 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


In [11]:
# 查看任务是否执行完成
print(str(handler.is_done))

<bound method WorkflowHandler.is_done of <workflows.handler.WorkflowHandler object at 0x36c216b90>>


In [12]:
# 取消任务
print(handler.cancel())

None
